# Phase 0 — First Look & Structural Verification
## 2021 Synthetic Integrated Services Data, Allegheny County DHS

**Approach.** Before analysis, I verify that the file matches what its creators documented
(Urban Institute *Synthetic Data User Guide*; *Generating a Fully Synthetic Human Services Dataset*, 2023).
Three tiers of checking, kept deliberately distinct:

| Tier | What | Handling |
|---|---|---|
| Mathematical invariants | e.g., probabilities ∈ [0,1] | `assert` — violation is a bug |
| Document-derived properties | claims the creators published | verify → report pass/fail, cite source |
| Empirical observations | anything the docs don't state | observe → interpret → let it shape next steps |

**Expectations from the documentation (before touching the data):**
1. Grain: one row = one person × one service × one month, 2021 only *(User Guide, "About the Data")*
2. Exactly 22 service categories *(User Guide service list)*
3. Excluded variables present but empty: geography, DOB/DOD, gender identity, sexual orientation, legal sex, employment, veteran flag *(User Guide, "Excluded Variables")*
4. Demographics constant across a person's rows; June-2022 snapshot *(User Guide)*
5. Homicides, suicides, overdoses, AND mental health crises occur at most once per person *(GEN, Stage One)* — a set of FOUR, distinct from the asterisked incident set (jail/homicides/overdoses/suicides) in the UG footnote
6. Child-only services capped at age ≤ 18 *(Generating…, Stage One constraints)*
7. Person-level total service-months cluster at multiples of 12; income supports is the most common service *(User Guide, Limitations; Generating…, synthesis order)*

Anything not on this list that the data shows me is an **observation**, to be interpreted, not assumed.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path

# Paths relative to the notebooks/ directory
DATA = Path("../data")

SRC = DATA / "dhs_service_records_synthesized_final.csv"
PARQUET = DATA / "person_level.parquet"
SVC_JSON = DATA / "service_columns.json"

print(f"Source: {SRC}")

con = duckdb.connect()

# Preview the data
sample = pd.read_csv(SRC, nrows=5)
sample

Source: ..\data\dhs_service_records_synthesized_final.csv


,synthetic_data,MCI_UNIQ_ID,CALDR_YR,DATE_OF_EVENT,service,GEO_AREA,age,DOB,DOD,GENDER,GENDER_IDENTITY,SEX_ORIENT,LEGAL_SEX,RACE,ETHNICITY,LIVING_ARRANGEMENT,EMPLOYMENT_STATUS,MARITAL_STATUS,EDUCATION_LEVEL,VETERAN_FLAG
0,SYNTHETIC DATA,1626,2021,01/31/2021,Children_Attending_Early_Childhood_Programs_Ma...,NaN,5,NaN,NaN,1~Male,NaN,NaN,NaN,1~White,99~Unknown,NaN,NaN,99~Unknown,9-12~High School (grade 9-12),NaN
1,SYNTHETIC DATA,1626,2021,02/28/2021,Children_Attending_Early_Childhood_Programs_Ma...,NaN,5,NaN,NaN,1~Male,NaN,NaN,NaN,1~White,99~Unknown,NaN,NaN,99~Unknown,9-12~High School (grade 9-12),NaN
2,SYNTHETIC DATA,1626,2021,03/31/2021,Children_Attending_Early_Childhood_Programs_Ma...,NaN,5,NaN,NaN,1~Male,NaN,NaN,NaN,1~White,99~Unknown,NaN,NaN,99~Unknown,9-12~High School (grade 9-12),NaN
3,SYNTHETIC DATA,1626,2021,04/30/2021,Children_Attending_Early_Childhood_Programs_Ma...,NaN,5,NaN,NaN,1~Male,NaN,NaN,NaN,1~White,99~Unknown,NaN,NaN,99~Unknown,9-12~High School (grade 9-12),NaN
4,SYNTHETIC DATA,1626,2021,05/31/2021,Children_Attending_Early_Childhood_Programs_Ma...,NaN,5,NaN,NaN,1~Male,NaN,NaN,NaN,1~White,99~Unknown,NaN,NaN,99~Unknown,9-12~High School (grade 9-12),NaN


### Check 1 — Grain and schema
The columns should match the documented layout. The repeated `MCI_UNIQ_ID` with changing
`DATE_OF_EVENT` in the sample is the service-month grain the User Guide describes.
`MCI` = Master Client Index — the warehouse's person identifier, per the Data Warehouse report.

In [2]:
row_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{SRC}')").fetchone()[0]
person_count = con.execute(f"SELECT COUNT(DISTINCT MCI_UNIQ_ID) FROM read_csv_auto('{SRC}')").fetchone()[0]
years = con.execute(f"SELECT DISTINCT CALDR_YR FROM read_csv_auto('{SRC}')").fetchall()
svc_count = con.execute(f"SELECT COUNT(DISTINCT service) FROM read_csv_auto('{SRC}')").fetchone()[0]
print(f"Rows: {row_count:,}  |  Distinct persons: {person_count:,}  (rows >> persons ⇒ long/event grain)")
print(f"Years present: {years}   → doc expectation: 2021 only")
print(f"Distinct services: {svc_count}  → doc expectation: 22")

Rows: 7,116,134  |  Distinct persons: 533,799  (rows >> persons ⇒ long/event grain)
Years present: [(2021,)]   → doc expectation: 2021 only
Distinct services: 22  → doc expectation: 22


**Interpretation.** Row count vastly exceeds person count — consistent with service-month grain, not
one-row-per-person. Year and service-count checks against expectations 1–2: if either fails, stop —
wrong file or wrong vintage. The person count itself is *not* in the documentation, so it is recorded
here as an **observation**, and becomes my denominator for everything person-level downstream.

In [3]:
# Check 2 -- grain uniqueness: is one row really one person x service x month?
# Everything downstream depends on this. The pivot below counts rows per (person, service) and
# reads that count as MONTHS RECEIVED; duplicated rows would silently inflate every service-month
# count, every co-occurrence cell, and the multiples-of-12 clustering check.
#
# Two checks, deliberately different in severity (CLAUDE.md: assert invariants, report documented
# properties):
#   HARD RAISE  COUNT(DISTINCT date) > 12 -- arithmetically impossible. Check 1 established the
#               file is 2021-only and [GEN] documents event dates as month-end, so a calendar year
#               offers exactly 12 distinct values. Exceeding it means a parse/filter bug, not data.
#   REPORT      COUNT(*) > 12, and COUNT(*) > COUNT(DISTINCT date) -- these mean duplicate rows,
#               a discrepancy with the [UG] grain claim and an observation to log, not a crash.
grain = con.execute(f"""
  SELECT MAX(n_rows) AS max_rows_per_person_service,
         MAX(n_months) AS max_distinct_months,
         SUM(CASE WHEN n_rows > 12 THEN 1 ELSE 0 END) AS ps_over_12_rows,
         SUM(CASE WHEN n_months > 12 THEN 1 ELSE 0 END) AS ps_over_12_months,
         SUM(CASE WHEN n_rows > n_months THEN 1 ELSE 0 END) AS ps_with_duplicate_rows
  FROM (SELECT MCI_UNIQ_ID, service, COUNT(*) AS n_rows,
               COUNT(DISTINCT DATE_OF_EVENT) AS n_months
        FROM read_csv_auto('{SRC}') GROUP BY MCI_UNIQ_ID, service)""").df().iloc[0]
print(grain.to_string())

if grain.ps_over_12_months > 0:
    raise ValueError(
        f"{grain.ps_over_12_months:,} (person, service) groups have more than 12 distinct "
        f"DATE_OF_EVENT values (max seen: {grain.max_distinct_months}).\n"
        "A 2021-only file has exactly 12 month-end dates, so this cannot be data -- it means the "
        "year filter, the date parse, or the source file is wrong. Stop and diagnose before "
        "trusting any count below.")

print(f"\nInvariant OK: no (person, service) exceeds 12 distinct month-end dates.")
if grain.ps_with_duplicate_rows > 0:
    print(f"REVIEW: {grain.ps_with_duplicate_rows:,} (person, service) groups have more rows than")
    print(f"  distinct months -- duplicate rows. The pivot's count-as-months reading is INVALID")
    print(f"  until resolved. Discrepancy with the grain claim [UG, 'About the Data'] -> findings_log.md.")
else:
    print("PASS: one row per person x service x month; count-as-months is a valid reading")
    print("  [UG, 'About the Data'].")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

max_rows_per_person_service    12.0
max_distinct_months            12.0
ps_over_12_rows                 0.0
ps_over_12_months               0.0
ps_with_duplicate_rows          0.0

Invariant OK: no (person, service) exceeds 12 distinct month-end dates.
PASS: one row per person x service x month; count-as-months is a valid reading
  [UG, 'About the Data'].


In [2]:
# Check 3 -- excluded variables should carry no real data (doc expectation 3)
# The User Guide's prose describes these as '"N/A" (null)'. Prose is not bytes: a column can be
# SQL NULL, or a constant sentinel string, and COUNT(DISTINCT) cannot tell those apart (it ignores
# NULLs, so all-null gives 0 and a constant string gives 1). Note also that pandas coerces bare
# 'NA' to NaN on read while DuckDB sees the literal string -- so the two engines disagree about
# this file. Report what is actually present; assume no sentinel.
excluded = ['GEO_AREA','DOB','DOD','GENDER_IDENTITY','SEX_ORIENT','LEGAL_SEX',
            'EMPLOYMENT_STATUS','VETERAN_FLAG']

def column_contents(col, src=SRC):
    """Return (n_rows, n_null, n_distinct_nonnull, sample_values) for one column."""
    q = f"""SELECT COUNT(*) AS n_rows,
                   COUNT(*) - COUNT({col}) AS n_null,
                   COUNT(DISTINCT {col}) AS n_distinct_nonnull
            FROM read_csv_auto('{SRC}')"""
    n_rows, n_null, n_distinct = con.execute(q).fetchone()
    vals = [r[0] for r in con.execute(
        f"SELECT DISTINCT {col} FROM read_csv_auto('{SRC}') "
        f"WHERE {col} IS NOT NULL LIMIT 5").fetchall()]
    return n_rows, n_null, n_distinct, vals

print("Excluded variables -- actual contents:\n")
report = []
for c in excluded:
    n_rows, n_null, n_distinct, vals = column_contents(c)
    if n_null == n_rows:
        state = "all NULL"
    elif n_distinct == 1:
        state = f"constant non-null sentinel {vals[0]!r}"
    else:
        state = f"CARRIES REAL VALUES ({n_distinct} distinct)"
    report.append((c, n_null, n_distinct, vals[0] if vals else None))
    print(f"  {c:20s} nulls={n_null:>10,}/{n_rows:,}  distinct_non_null={n_distinct:<4} {state}")

carries_data = [c for c, _, nd, _ in report if nd > 1]
sentinels = sorted({r[3] for r in report if r[2] == 1})
print()
if carries_data:
    print(f"REVIEW: an 'excluded' variable carries real values -> {carries_data}. Re-read the data dictionary.")
else:
    print("PASS: no excluded variable carries analysable values [UG, 'Excluded Variables'].")
if sentinels:
    print(f"DISCREPANCY TO LOG: doc prose says '\"N/A\" (null)' (documented_facts.md, Excluded Variables),")
    print(f"  but the file uses a non-null sentinel: {sentinels}. Record in findings_log.md;")
    print(f"  do not amend documented_facts.md without the [OBSERVED: ...] tag.")

Excluded variables -- actual contents:



  GEO_AREA             nulls=         0/7,116,134  distinct_non_null=1    constant non-null sentinel 'NA'


  DOB                  nulls=         0/7,116,134  distinct_non_null=1    constant non-null sentinel 'NA'


  DOD                  nulls=         0/7,116,134  distinct_non_null=1    constant non-null sentinel 'NA'


  GENDER_IDENTITY      nulls=         0/7,116,134  distinct_non_null=1    constant non-null sentinel 'NA'


  SEX_ORIENT           nulls=         0/7,116,134  distinct_non_null=1    constant non-null sentinel 'NA'


  LEGAL_SEX            nulls=         0/7,116,134  distinct_non_null=1    constant non-null sentinel 'NA'


  EMPLOYMENT_STATUS    nulls=         0/7,116,134  distinct_non_null=1    constant non-null sentinel 'NA'


  VETERAN_FLAG         nulls=         0/7,116,134  distinct_non_null=1    constant non-null sentinel 'NA'

PASS: no excluded variable carries analysable values [UG, 'Excluded Variables'].
DISCREPANCY TO LOG: doc prose says '"N/A" (null)' (documented_facts.md, Excluded Variables),
  but the file uses a non-null sentinel: ['NA']. Record in findings_log.md;
  do not amend documented_facts.md without the [OBSERVED: ...] tag.


**Why this matters analytically.** Confirming these columns are empty is not pedantry — it fixes the
demographic lens list for the entire project: the usable demographics are exactly age, gender, race,
ethnicity, marital status, and education. No geographic analysis is possible, by design.

In [5]:
# Check 3b -- LIVING_ARRANGEMENT: available or not? (open decision M8)
# The two sources disagree: [UG, "About the Data"] lists living arrangement among the per-row
# demographics, while [GEN] discusses it under variable exclusions. CLAUDE.md treats its
# availability as UNCONFIRMED until verified here. Report only -- never assert, and do not
# pick a side: surface the disagreement.
#
# Uses the same content-aware logic as Check 3 deliberately. A bare COUNT(DISTINCT) returns 1
# for a constant sentinel, which a reader would misread as "contains data" and resolve M8
# backwards.
LA = 'LIVING_ARRANGEMENT'
cols = [c.upper() for c in pd.read_csv(SRC, nrows=0).columns]

if LA not in cols:
    print(f"{LA}: COLUMN ABSENT from the file entirely.")
    verdict = "absent -> unusable"
else:
    n_rows, n_null, n_distinct, vals = column_contents(LA)
    print(f"{LA}: rows={n_rows:,}  nulls={n_null:,}  distinct_non_null={n_distinct}")
    print(f"  sample non-null values: {vals}")
    if n_null == n_rows:
        verdict = "present but entirely NULL -> unusable"
    elif n_distinct <= 1:
        verdict = f"present but constant sentinel {vals[0]!r} (no information) -> unusable"
    else:
        verdict = f"CARRIES {n_distinct} distinct values -> potentially usable"

print(f"\nM8 (living arrangement): {verdict}")
print("Sources disagree -- [UG] lists it as a row demographic; [GEN] discusses it under exclusions.")
print("Log this result in findings_log.md against that conflict (CLAUDE.md: surface, do not override).")
print("Whether to add it to the person-level pivot is a separate decision, not taken here.")

LIVING_ARRANGEMENT: rows=7,116,134  nulls=0  distinct_non_null=1
  sample non-null values: ['NA']

M8 (living arrangement): present but constant sentinel 'NA' (no information) -> unusable
Sources disagree -- [UG] lists it as a row demographic; [GEN] discusses it under exclusions.
Log this result in findings_log.md against that conflict (CLAUDE.md: surface, do not override).
Whether to add it to the person-level pivot is a separate decision, not taken here.


In [6]:
# Check 4 -- demographics constant within person (doc expectation 4)
# All six documented per-row demographics, reported PER VARIABLE rather than as one aggregate
# count: postprocessing forced under-18 marital-status/education values to "unknown" [GEN,
# "Postprocessing"], so a violation in those two has a documented explanation that a violation
# in age/gender/race/ethnicity would not. Report-only [UG, "About the Data"] -- a violation is a
# discrepancy to surface, not a crash.
demographics = ['age', 'GENDER', 'RACE', 'ETHNICITY', 'MARITAL_STATUS', 'EDUCATION_LEVEL']

viol = con.execute(f"""
  SELECT {', '.join(f'SUM(CASE WHEN d_{c} > 1 THEN 1 ELSE 0 END) AS "{c}"' for c in demographics)}
  FROM (
    SELECT MCI_UNIQ_ID,
           {', '.join(f'COUNT(DISTINCT {c}) AS d_{c}' for c in demographics)}
    FROM read_csv_auto('{SRC}') GROUP BY MCI_UNIQ_ID)""").df().T
viol.columns = ['persons_with_multiple_values']
print("Persons whose value for a demographic changes across their rows:\n")
print(viol)

bad = viol[viol.persons_with_multiple_values > 0]
print()
if bad.empty:
    print("PASS: all six demographics are constant per person [UG, 'About the Data'].")
else:
    print(f"REVIEW: {list(bad.index)} vary within person -- doc says demographics are constant.")
    print("  Note: MARITAL_STATUS / EDUCATION_LEVEL have a documented postprocessing explanation")
    print("  (under-18 values forced to 'unknown') [GEN]; the other four do not. Log the")
    print("  discrepancy in findings_log.md; note any_value() in the pivot silently picks one.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Persons whose value for a demographic changes across their rows:

                 persons_with_multiple_values
age                                       0.0
GENDER                                    0.0
RACE                                      0.0
ETHNICITY                                 0.0
MARITAL_STATUS                            0.0
EDUCATION_LEVEL                           0.0

PASS: all six demographics are constant per person [UG, 'About the Data'].


In [7]:
# Check 5 -- incident variables at most once per person (doc expectation 5)
#
# The four service names are RESOLVED from the data, not hardcoded. Hardcoding them made this
# check able to pass vacuously: if no literal matched, `WHERE service IN (...)` returned an empty
# frame, which reads as "nothing wrong" rather than "the check never ran".
#
# Match on distinguishing tokens rather than literal documented names, because the doc phrasing
# and the data values differ -- [UG] writes "Individuals receiving services for a mental health
# crisis", "Individuals identified as homeless", "Individuals in the Allegheny County jail".
# Resolving documented names literally would fail. Do not "simplify" these back to literals.
#
# 'crisis|crises' is deliberately narrow: a bare 'mental health' pattern would also match the
# general mental-health-services category, which is NOT once-per-person. Requiring exactly one
# match per pattern is what catches that.
PATTERNS = {'homicides': r'homicide', 'suicides': r'suicide',
            'overdoses': r'overdose', 'mental health crises': r'crisis|crises'}

actual_services = [r[0] for r in con.execute(
    f"SELECT DISTINCT service FROM read_csv_auto('{SRC}') ORDER BY 1").fetchall()]

resolved, problems = {}, []
for label, pat in PATTERNS.items():
    hits = [s for s in actual_services if re.search(pat, s, re.I)]
    if len(hits) == 1:
        resolved[label] = hits[0]
    else:
        problems.append(f"  {label!r}: pattern {pat!r} matched {len(hits)} service(s) {hits}")

# HARD RAISE: this guards against a lookup/code bug, not a fact about the data. If it fires, every
# number below it is vacuous. `raise` rather than `assert` -- survives python -O, carries a message.
if problems:
    raise ValueError(
        "Cannot resolve the four documented once-per-person services [GEN, Stage One] to actual "
        "data values.\n" + "\n".join(problems) +
        f"\n\nActual services present ({len(actual_services)}):\n  " + "\n  ".join(actual_services))

print("Resolved once-per-person services (doc label -> actual data value):")
for k, v in resolved.items():
    print(f"  {k:22s} -> {v}")

# REPORT: the once-per-person property itself is a documented claim [GEN, Stage One]. A failure
# here is a discrepancy with the source to surface and log, not a crash.
in_list = ','.join("'" + s.replace("'", "''") + "'" for s in resolved.values())
per_person = con.execute(f"""
  SELECT service, MAX(cnt) AS max_rows_per_person, COUNT(*) AS n_persons FROM (
    SELECT service, MCI_UNIQ_ID, COUNT(*) cnt FROM read_csv_auto('{SRC}')
    WHERE service IN ({in_list})
    GROUP BY service, MCI_UNIQ_ID) GROUP BY service ORDER BY service""").df()
print()
print(per_person)

if len(per_person) != len(resolved):
    print(f"\nREVIEW: expected {len(resolved)} services in the result, got {len(per_person)}.")
over = per_person[per_person.max_rows_per_person > 1]
if over.empty:
    print("\nPASS: all four appear at most once per person [GEN, 'Stage One'].")
else:
    print(f"\nREVIEW: {list(over.service)} exceed one row per person -- discrepancy with")
    print("  [GEN, 'Stage One'], which documents this constraint as enforced in synthesis.")
    print("  Surface it in findings_log.md; do not silently override either side (CLAUDE.md).")

Resolved once-per-person services (doc label -> actual data value):
  homicides              -> Homicides
  suicides               -> Suicides
  overdoses              -> Overdoses
  mental health crises   -> Mental_Health_Crises



                service  max_rows_per_person  n_persons
0             Homicides                    1        172
1  Mental_Health_Crises                    1       7245
2             Overdoses                    1       1319
3              Suicides                    1       1177

PASS: all four appear at most once per person [GEN, 'Stage One'].


**Interpretation guide.** The source (GEN, Stage One) documents FOUR services as appearing only once
per individual in the confidential data, and this constraint was enforced in synthesis: homicides,
suicides, overdoses, AND mental health crises. So all four should show max 1 row per person here.
(Note: this once-per-person set is DISTINCT from the asterisked "incidents, not services" set in the
UG footnote, which is jail + homicides + overdoses + suicides — mental health crises is once-per-person
but not asterisked; jail is asterisked but not once-per-person.) Operationally, homicide/suicide/overdose
are often fatal, which truncates that person's service year and mechanically deflates their other service
counts — any later finding about these populations must carry that caveat. Record the observed max for
each; if any exceeds 1, that is a discrepancy with the source to note explicitly.

In [ ]:
# Check 6 — age eligibility on child-targeted services (doc expectation 6)
#
# REPORT-ONLY BY DESIGN. This check never raises. [GEN, "Stage One"] says age eligibility
# constraints were applied per-service from individual eligibility rules and gives "children
# capped at age 18" only as an EXAMPLE ("e.g."), naming DHS-funded out-of-school programs. It
# does not state child welfare's cap, so an over-18 max here is not provably a violation -- it is
# a discrepancy to surface and carry, not a bug to crash on.
#
# Child welfare's real age cap is UNVERIFIED pending the source [DW] PDF. The published data
# dictionary cannot settle it either: its age description is erroneous boilerplate ("Age of
# Decedent for the morgue autopsy case"). See the age-24 entry in docs/findings_log.md.
CHILD_AGE_EXAMPLE_CAP = 18   # the [GEN] example, NOT a documented universal cap

child_svcs = con.execute(f"""SELECT service, MIN(age) min_age, MAX(age) max_age
   FROM read_csv_auto('{SRC}') WHERE service LIKE 'Child%' GROUP BY service ORDER BY service""").df()
print(child_svcs)

over_cap = child_svcs[child_svcs.max_age > CHILD_AGE_EXAMPLE_CAP]
print()
if over_cap.empty:
    print(f"PASS: every child-targeted service stays at or below age {CHILD_AGE_EXAMPLE_CAP} "
          f"[GEN, 'Stage One' example].")
else:
    for _, r in over_cap.iterrows():
        print(f"REVIEW (already logged open): {r.service} max age {r.max_age} exceeds the "
              f"age-{CHILD_AGE_EXAMPLE_CAP} example in [GEN]; conflicts with [DW #7] "
              f"(child welfare 'youth 18 or younger'); see findings_log.md age-24 entry. "
              f"Not resolved here.")
    print(f"\n  {len(child_svcs) - len(over_cap)} of {len(child_svcs)} child services are within "
          f"the example cap. Report-only: nothing is asserted and nothing is promoted to")
    print(f"  documented_facts.md until the [DW] source is checked.")

In [9]:
# Build person-level table (one row per person, months-count per service) for expectation 7
services = [r[0] for r in con.execute(f"SELECT DISTINCT service FROM read_csv_auto('{SRC}') ORDER BY 1").fetchall()]
pivot = ", ".join(f'COUNT(*) FILTER (WHERE service=\'{s}\') AS "{s}"' for s in services)
con.execute(f"""COPY (
  WITH counts AS (SELECT MCI_UNIQ_ID person_id, {pivot} FROM read_csv_auto('{SRC}') GROUP BY 1),
  demo AS (SELECT MCI_UNIQ_ID person_id, any_value(age) age, any_value(GENDER) gender,
           any_value(RACE) race, any_value(ETHNICITY) ethnicity,
           any_value(MARITAL_STATUS) marital_status, any_value(EDUCATION_LEVEL) education_level
           FROM read_csv_auto('{SRC}') GROUP BY 1)
  SELECT d.*, c.* EXCLUDE(person_id) FROM demo d JOIN counts c USING(person_id)
) TO '{PARQUET}' (FORMAT PARQUET)""")

# Persist the service column names explicitly. Phase 1 reads this instead of inferring services by
# excluding a hardcoded demographic list -- exclusion silently promotes any newly added column into
# the co-occurrence matrix as though it were a service. Written from the same `services` list that
# built the pivot (not a re-query) so the two cannot drift.
with open(SVC_JSON, 'w') as f:
    json.dump(services, f, indent=2)
print(f"Wrote {SVC_JSON} with {len(services)} service columns.")
print(f"  Doc expectation: 22 [UG, 'About the Data'] -- "
      f"{'match' if len(services) == 22 else 'MISMATCH, investigate before Phase 1'}")

pl = pd.read_parquet(PARQUET)
pl['total_service_months'] = pl[services].sum(axis=1)
pl['distinct_services'] = (pl[services]>0).sum(axis=1)
print(pl.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote ..\data\service_columns.json with 22 service columns.
  Doc expectation: 22 [UG, 'About the Data'] -- match


(533799, 31)


In [10]:
# Check 7 — clustering at multiples of 12; most common service
vc = pl.total_service_months.value_counts()
top5 = vc.head(5)
print("Most frequent total service-month values:\n", top5)
mode_val = vc.idxmax()
mult12_at_top = mode_val % 12 == 0
prev = (pl[services]>0).mean().sort_values(ascending=False)
print(f"\nModal total = {mode_val} ({'multiple of 12 ✔' if mult12_at_top else 'NOT a multiple of 12 — investigate'})")
print(f"Share of persons at modal value: {vc.max()/len(pl):.1%}  ← empirical observation, not a doc number")
print(f"\nMost common service: {prev.index[0]}  ({prev.iloc[0]:.1%} of persons)")
print("Doc expectation: clustering at multiples of 12; income supports most common (Generating…, synthesis order)")

Most frequent total service-month values:
 total_service_months
12    328472
4      13313
1      12682
36     10961
2      10451
Name: count, dtype: int64

Modal total = 12 (multiple of 12 ✔)
Share of persons at modal value: 61.5%  ← empirical observation, not a doc number

Most common service: Individuals_Receiving_Income_Supports  (94.1% of persons)
Doc expectation: clustering at multiples of 12; income supports most common (Generating…, synthesis order)


### First-look conclusions and decisions for Phase 1

**Structure — verified, with one open item.** Grain, 22 services, 2021-only, excluded-variable
emptiness, constant demographics, and the four once-per-person incidents all matched the
documentation. Child age caps hold for four of five child services (early childhood 0–5, early
intervention 0–3, out-of-school 0–17, children in care 0–17); `Children_Receiving_Child_Welfare_Services`
showed max age 24, which conflicts with [DW #7] and is logged as an **OPEN discrepancy**
(`findings_log.md`), not a confirmation. The file otherwise behaves as its creators described, so
downstream analysis inherits their validation work.

**Key observations.** N = 533,799 persons (7,116,134 rows); modal total service-months = 12, held by
61.5% of persons; dominant service = income supports at 94.1% of persons. Because one service touches
94% of everyone, a raw conditional co-occurrence rate toward it would look strong for almost any
service purely as an artifact of prevalence — so Phase 1 reports **lift** (which divides out each
service's base rate) alongside the conditional percentage, and applies a **minimum-support floor**
(in the spirit of the county's <6 small-cell suppression) before reporting any pair. This is the
evidence behind decision M4. N = 533,799 also fixes the documented 0.5%-of-individuals fidelity tier
at **2,669 shared persons** — the cutoff the pending rare-pair decision (M6) is measured against.

**Standing caveats for Phase 1** (from [UG, Limitations]): overrepresented populations — jail, Child
Welfare families, homeless, homicide/suicide/overdose — get relative/shape claims only, never
prevalence claims. Older adults are undercounted. Separately, a fatal-incident truncation caveat
(documented/inferred, **not measured here**): homicide/suicide/overdose are often fatal, ending the
person's service year and mechanically deflating their other service counts — any finding about these
populations must carry it.